In [11]:
import numpy as np
import pandas as pd
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import RandomizedSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import roc_auc_score

In [12]:
file_path = 'Data/dane.csv'

heart_test = pd.read_csv('Data/heart_test.csv')
heart_train = pd.read_csv('Data/heart_train.csv')
mushrooms_test = pd.read_csv('Data/mushrooms_test.csv')
mushrooms_train = pd.read_csv('Data/mushrooms_train.csv')
rice_test = pd.read_csv('Data/rice_test.csv')
rice_train = pd.read_csv('Data/rice_train.csv')
wine_test = pd.read_csv('Data/wine_test.csv')
wine_train = pd.read_csv('Data/wine_train.csv')

datasets = {
    "heart": (heart_train, heart_test),
    "mushrooms": (mushrooms_train, mushrooms_test),
    "rice": (rice_train, rice_test),
    "wine": (wine_train, wine_test)
}

In [13]:
n_random = 100
np.random.seed(42)

from scipy.stats import randint

param_dist_knn = {
    'n_neighbors': randint(1, 50),           # liczba sąsiadów w zakresie [1, 50)
    'weights': ['uniform', 'distance'],      # sposób ważenia sąsiadów
    'p': randint(1, 3),                      # 1 = Manhattan, 2 = Euklides
    'metric': ['minkowski']                 # można dodać inne, np. 'euclidean', 'manhattan'
}



In [14]:
all_results = []

for name, (train, test) in datasets.items():
    print(f"Trenuję model KNN dla: {name}")

    X_train, y_train = train.iloc[:, :-1], train.iloc[:, -1]
    X_test, y_test = test.iloc[:, :-1], test.iloc[:, -1]

    model = KNeighborsClassifier()

    search = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_dist_knn,
        n_iter=50,
        scoring='roc_auc',
        cv=5,
        verbose=1,
        random_state=42,
        n_jobs=-1
    )

    search.fit(X_train, y_train)

    cv_results = pd.DataFrame(search.cv_results_)

    for i, params in enumerate(search.cv_results_['params']):
        model = KNeighborsClassifier(**params)
        model.fit(X_train, y_train)
        y_proba = model.predict_proba(X_test)[:, 1]
        test_auc = roc_auc_score(y_test, y_proba)

        all_results.append({
            "dataset": name,
            "params": params,
            "cv_roc_auc": cv_results.loc[i, 'mean_test_score'],
            "test_roc_auc": test_auc
        })

results_df = pd.DataFrame(all_results)

results_df.to_csv("knn.csv", index=False)

Trenuję model KNN dla: heart
Fitting 5 folds for each of 50 candidates, totalling 250 fits
Trenuję model KNN dla: mushrooms
Fitting 5 folds for each of 50 candidates, totalling 250 fits
Trenuję model KNN dla: rice
Fitting 5 folds for each of 50 candidates, totalling 250 fits
Trenuję model KNN dla: wine
Fitting 5 folds for each of 50 candidates, totalling 250 fits


In [15]:
wyniki_knn = pd.read_csv('knn.csv')

In [16]:
best_per_dataset = (
    wyniki_knn.sort_values(by=["dataset", "test_roc_auc"], ascending=[True, False]).groupby("dataset", as_index=False).first()
)
params_df = best_per_dataset["params"].apply(pd.Series)

aggregated_params = {}

for col in params_df.columns:
    if pd.api.types.is_numeric_dtype(params_df[col]):
        aggregated_params[col] = params_df[col].mean()
    else:
        aggregated_params[col] = params_df[col].mode().iloc[0]  # najczęstsza wartość

mean_params = pd.Series(aggregated_params)

In [17]:
mean_params

0    {'metric': 'minkowski', 'n_neighbors': 15, 'p'...
dtype: object

In [20]:
# 1. Rozpakuj kolumnę 'params' na osobne kolumny
params_df = results_df['params'].apply(pd.Series)
full_df = pd.concat([results_df.drop(columns='params'), params_df], axis=1)

mean_results = []

# 2. Iteruj po zbiorach danych
for name, (train, test) in datasets.items():


    X_train, y_train = train.iloc[:, :-1], train.iloc[:, -1]
    X_test, y_test = test.iloc[:, :-1], test.iloc[:, -1]

    # 2b. Oblicz średnie hiperparametry dla danego zbioru
    subset_df = full_df[full_df["dataset"] == name]

    # Tylko te hiperparametry, które są numeryczne
    mean_params = {
        "n_neighbors": int(round(subset_df["n_neighbors"].mean())),
        "p": int(round(subset_df["p"].mean())),
        "weights": subset_df["weights"].mode()[0],
        "metric": subset_df["metric"].mode()[0]
    }

    # 2c. Trenuj i testuj model z uśrednionymi parametrami
    model = KNeighborsClassifier(**mean_params)
    model.fit(X_train, y_train)
    y_proba = model.predict_proba(X_test)[:, 1]
    mean_auc = roc_auc_score(y_test, y_proba)

    mean_results.append({
        "dataset": name,
        "mean_test_roc_auc": mean_auc,
        "used_params": mean_params
    })

# 3. Wyniki do DataFrame
mean_df = pd.DataFrame(mean_results)

In [21]:
results_df = results_df.merge(mean_df, on="dataset")
results_df["diff_from_mean"] = results_df["mean_test_roc_auc"] - results_df["test_roc_auc"]
results_df

results_df.sort_values(by="diff_from_mean",ascending=True).head(20)

results_df.to_csv("results_knn.csv", index=False)